# Transform drivers Data

1. Read bronze _drivers_ table
2. Keep only the columns required for analytics (Drop _url_ column).
3. Standardise column names using snake_case (_driverId_ -> _driver_id_ for example)
4. Concatenate name.givenName and name.familyName to create a new column driver_name and transform the value to Title Case.
5. Remove duplicate records
6. Transform values of columns _nationality_ to Title Case
7. Write the transformed data to silver _drivers_ table.

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
from pyspark.sql import functions as F

### Step 1 - Read bronze _drivers_ table

In [0]:
# Methode 1 that allows options since it's an API call:
# drivers_df = spark.read.table(bronze_table)

In [0]:
# Method 2 using a method with no additionnal options
drivers_df = spark.table(bronze_table)

In [0]:
display(drivers_df)

**2. Keep only the columns required for analytics (Drop _url_ column).**

In [0]:
drivers_dropped_df = drivers_df.drop("url")

In [0]:
display(drivers_dropped_df)

**3. Standardise column names using snake_case (_circuitId_ -> _circuit_id_ for example) &
4. Rename columns to make them more meaningful (_lat_ -> latitude for example)**

In [0]:
# ANCIENNE VERSION!!
# drivers_renamed_df = (
#     drivers_selected_df
#     .withColumnRenamed("circuitId", "circuit_id")
#     .withColumnRenamed("circuitName", "circuit_name")
#     .withColumnRenamed("lat", "latitude")
#     .withColumnRenamed("long","longitude")                       
# )

In [0]:
drivers_renamed_df = (
    drivers_dropped_df
    .withColumnsRenamed({
        "dateOfBirth": "date_of_birth",
        "driverId": "driver_id"
    })                     
)

In [0]:
display(drivers_renamed_df)

**4. Concatenate to create a new column driver_name:**

In [0]:
drivers_concatenated_df = (
    drivers_renamed_df
        .withColumn("driver_name", 
                    F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))))
        .drop("name")
)

In [0]:
display(drivers_concatenated_df)

**5. Remove duplicate records**

In [0]:
# drivers_distinct_df = drivers_valid_df.distinct()

In [0]:
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])

In [0]:
display(drivers_distinct_df)

**6. Transform values of column nationality to Title Case**

In [0]:
drivers_final_df = (
    drivers_distinct_df
        .withColumn("nationality", F.initcap(F.col("nationality")))
)

In [0]:
display(drivers_final_df)

**7. Write the transformed data to silver drivers table.**

In [0]:
(
    drivers_final_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))